In [1]:
from flask import Flask, render_template_string, request, redirect, jsonify
from flask_sqlalchemy import SQLAlchemy
from flask_cors import CORS
import threading, os, joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import numpy as np

# =========================
# Flask App
app = Flask(__name__)
CORS(app)
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///staff.db'
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False

db = SQLAlchemy(app)

# =========================
# ML Model Files
ATTENDANCE_MODEL_FILE = "staff_model.pkl"
ATTRITION_MODEL_FILE = "attrition_model.pkl"

# =========================
# Staff Model
class Staff(db.Model):
    id = db.Column(db.String, primary_key=True)
    name = db.Column(db.String, nullable=False)
    days_present = db.Column(db.Integer, default=0)
    days_absent = db.Column(db.Integer, default=0)
    month = db.Column(db.String, default="")
    base_salary = db.Column(db.Float, default=0)
    bonus = db.Column(db.Float, default=0)
    deductions = db.Column(db.Float, default=0)
    net_pay = db.Column(db.Float, default=0)
    leave_balance = db.Column(db.Integer, default=0)
    leaves_taken = db.Column(db.Integer, default=0)
    pending_requests = db.Column(db.Integer, default=0)
    kpi_score = db.Column(db.Integer, default=0)
    peer_review = db.Column(db.String, default="")
    manager_feedback = db.Column(db.String, default="")

# =========================
# ML Helpers
def train_attendance_model():
    staff_list = Staff.query.all()
    if not staff_list:
        return None
    X, y = [], []
    months = [s.month for s in staff_list]
    le = LabelEncoder()
    months_encoded = le.fit_transform(months) if months else [0]*len(staff_list)
    for i, s in enumerate(staff_list):
        X.append([s.base_salary, s.net_pay, s.days_present, s.days_absent, months_encoded[i]])
        y.append(1 if s.days_present >= s.days_absent else 0)
    if len(set(y)) < 2:
        return None
    model = LogisticRegression()
    model.fit(X, y)
    joblib.dump((model, le), ATTENDANCE_MODEL_FILE)
    return model, le

def load_attendance_model():
    if os.path.exists(ATTENDANCE_MODEL_FILE):
        return joblib.load(ATTENDANCE_MODEL_FILE)
    return train_attendance_model()

def train_attrition_model():
    staff_list = Staff.query.all()
    if not staff_list:
        return None
    X, y = [], []
    months = [s.month for s in staff_list]
    le = LabelEncoder()
    months_encoded = le.fit_transform(months) if months else [0]*len(staff_list)
    for i, s in enumerate(staff_list):
        X.append([s.days_present, s.days_absent, s.leave_balance, s.kpi_score, months_encoded[i]])
        y.append(1 if (s.days_present < s.days_absent or s.kpi_score < 50) else 0)
    if len(set(y)) < 2:
        return None
    model = LogisticRegression()
    model.fit(X, y)
    joblib.dump((model, le), ATTRITION_MODEL_FILE)
    return model, le

def load_attrition_model():
    if os.path.exists(ATTRITION_MODEL_FILE):
        return joblib.load(ATTRITION_MODEL_FILE)
    return train_attrition_model()

# =========================
# Helper to serialize staff
def serialize_staff(staff):
    return {
        "id": staff.id, "name": staff.name,
        "days_present": staff.days_present, "days_absent": staff.days_absent,
        "month": staff.month, "base_salary": staff.base_salary, "net_pay": staff.net_pay,
        "bonus": staff.bonus, "deductions": staff.deductions,
        "leave_balance": staff.leave_balance, "leaves_taken": staff.leaves_taken,
        "pending_requests": staff.pending_requests, "kpi_score": staff.kpi_score,
        "peer_review": staff.peer_review, "manager_feedback": staff.manager_feedback
    }

# =========================
# Dashboard
@app.route('/')
def index():
    staff_list = Staff.query.all()
    template = """
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>Staff Dashboard</title>
        <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    </head>
    <body class="bg-light">
    <div class="container py-4">
        <h1 class="mb-4 text-center">Staff Dashboard</h1>
        <div class="text-center mb-3">
            <a href="/add_staff" class="btn btn-primary">Add New Staff</a>
        </div>
        <div class="row row-cols-1 row-cols-md-2 row-cols-lg-3 g-4">
        {% for s in staff %}
            <div class="col">
                <div class="card h-100 shadow-sm">
                    <div class="card-header bg-primary text-white">
                        <h5>{{s.id}} - {{s.name}}</h5>
                    </div>
                    <div class="card-body">
                        <p><strong>Base Salary:</strong> {{s.base_salary}}</p>
                        <p><strong>Net Pay:</strong> {{s.net_pay}}</p>
                        <a href="/staff/{{s.id}}" class="btn btn-sm btn-success">View & Edit Modules</a>
                        <a href="/delete/{{s.id}}" class="btn btn-sm btn-danger">Delete</a>
                        <button class="btn btn-sm btn-primary" onclick="predictAttendance('{{s.id}}', this)">Attendance Prediction</button>
                        <button class="btn btn-sm btn-warning" onclick="predictAttrition('{{s.id}}', this)">Attrition Risk</button>
                        <div class="mt-2 prediction-result"></div>
                    </div>
                </div>
            </div>
        {% endfor %}
        </div>
    </div>
    <script>
    function predictAttendance(staffId, btn){
        const card = btn.closest('.card-body');
        const resultDiv = card.querySelector('.prediction-result');
        resultDiv.innerHTML = "Predicting attendance...";
        fetch(`/predict_attendance/${staffId}`)
        .then(r=>r.json()).then(data=>{
            if(data.error) resultDiv.innerHTML=`<span class="text-danger">${data.error}</span>`;
            else resultDiv.innerHTML=`<strong>Attendance: ${data.prediction}</strong> (Probability: ${data.probability})`;
        });
    }
    function predictAttrition(staffId, btn){
        const card = btn.closest('.card-body');
        const resultDiv = card.querySelector('.prediction-result');
        resultDiv.innerHTML += "<br>Predicting attrition...";
        fetch(`/predict_attrition/${staffId}`)
        .then(r=>r.json()).then(data=>{
            if(data.error) resultDiv.innerHTML+=`<br><span class="text-danger">${data.error}</span>`;
            else resultDiv.innerHTML+=`<br><strong>Attrition: ${data.prediction}</strong> (Probability: ${data.probability})`;
        });
    }
    </script>
    </body>
    </html>
    """
    staff_serialized = [serialize_staff(s) for s in staff_list]
    return render_template_string(template, staff=staff_serialized)

# =========================
# Add Staff

@app.route('/add_staff', methods=['GET', 'POST'])
def add_staff():
    if request.method == 'POST':
        staff_id = request.form['id']
        name = request.form['name']
        if Staff.query.get(staff_id):
            return f"<h3>Staff ID {staff_id} already exists!</h3><a href='/'>Back</a>"
        staff = Staff(id=staff_id, name=name)
        db.session.add(staff)
        db.session.commit()
        return redirect('/')
    template = """
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>Add Staff</title>
        <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    </head>
    <body class="bg-light">
        <div class="container py-4">
            <h1 class="mb-4">Add Staff</h1>
            <form method="post" class="card p-3 shadow-sm bg-white">
                <div class="mb-3">
                    <label>ID:</label>
                    <input type="text" name="id" class="form-control" required>
                </div>
                <div class="mb-3">
                    <label>Name:</label>
                    <input type="text" name="name" class="form-control" required>
                </div>
                <button class="btn btn-primary" type="submit">Add Staff</button>
                <a href="/" class="btn btn-secondary">Back</a>
            </form>
        </div>
    </body>
    </html>
    """
    return render_template_string(template)

@app.route('/staff/<staff_id>', methods=['GET','POST'])
def staff_detail(staff_id):
    staff = Staff.query.get(staff_id)
    if not staff:
        return "<h3>Staff Not Found</h3><a href='/'>Back to Dashboard</a>", 404

    if request.method == 'POST':
        module = request.form.get('module')
        if module == 'attendance':
            staff.days_present = int(request.form.get('days_present', staff.days_present))
            staff.days_absent = int(request.form.get('days_absent', staff.days_absent))
            staff.month = request.form.get('month', staff.month)
        elif module == 'payroll':
            staff.base_salary = float(request.form.get('base_salary', staff.base_salary))
            staff.net_pay = float(request.form.get('net_pay', staff.net_pay))
            staff.bonus = float(request.form.get('bonus', staff.bonus))
            staff.deductions = float(request.form.get('deductions', staff.deductions))
        elif module == 'leave':
            staff.leave_balance = int(request.form.get('leave_balance', staff.leave_balance))
            staff.leaves_taken = int(request.form.get('leaves_taken', staff.leaves_taken))
            staff.pending_requests = int(request.form.get('pending_requests', staff.pending_requests))
        elif module == 'performance':
            staff.kpi_score = int(request.form.get('kpi_score', staff.kpi_score))
            staff.peer_review = request.form.get('peer_review', staff.peer_review)
            staff.manager_feedback = request.form.get('manager_feedback', staff.manager_feedback)
        db.session.commit()
        return redirect(f'/staff/{staff_id}')

    template = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Staff {{staff.id}}</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
</head>
<body class="bg-light">
<div class="container py-4">
    <h1 class="mb-4">Staff {{staff.id}} - {{staff.name}}</h1>
    <a href="/" class="btn btn-secondary mb-3">Back to Dashboard</a>
    <div class="row row-cols-1 row-cols-md-2 g-4">

        <!-- Attendance Card -->
        <div class="col">
            <div class="card shadow-sm">
                <div class="card-header bg-info text-white">Attendance</div>
                <div class="card-body">
                    <form method="post">
                        <input type="hidden" name="module" value="attendance">
                        <div class="mb-2">Days Present: <input type="number" name="days_present" class="form-control" value="{{staff.days_present}}"></div>
                        <div class="mb-2">Days Absent: <input type="number" name="days_absent" class="form-control" value="{{staff.days_absent}}"></div>
                        <div class="mb-2">Month: <input type="text" name="month" class="form-control" value="{{staff.month}}"></div>
                        <button class="btn btn-info w-100">Update Attendance</button>
                    </form>
                </div>
            </div>
        </div>

        <!-- Payroll Card -->
        <div class="col">
            <div class="card shadow-sm">
                <div class="card-header bg-success text-white">Payroll</div>
                <div class="card-body">
                    <form method="post">
                        <input type="hidden" name="module" value="payroll">
                        <div class="mb-2">Base Salary: <input type="number" step="0.01" name="base_salary" class="form-control" value="{{staff.base_salary}}"></div>
                        <div class="mb-2">Net Pay: <input type="number" step="0.01" name="net_pay" class="form-control" value="{{staff.net_pay}}"></div>
                        <div class="mb-2">Bonus: <input type="number" step="0.01" name="bonus" class="form-control" value="{{staff.bonus}}"></div>
                        <div class="mb-2">Deductions: <input type="number" step="0.01" name="deductions" class="form-control" value="{{staff.deductions}}"></div>
                        <button class="btn btn-success w-100">Update Payroll</button>
                    </form>
                </div>
            </div>
        </div>

        <!-- Leave Card -->
        <div class="col">
            <div class="card shadow-sm">
                <div class="card-header bg-warning text-dark">Leave</div>
                <div class="card-body">
                    <form method="post">
                        <input type="hidden" name="module" value="leave">
                        <div class="mb-2">Leave Balance: <input type="number" name="leave_balance" class="form-control" value="{{staff.leave_balance}}"></div>
                        <div class="mb-2">Leaves Taken: <input type="number" name="leaves_taken" class="form-control" value="{{staff.leaves_taken}}"></div>
                        <div class="mb-2">Pending Requests: <input type="number" name="pending_requests" class="form-control" value="{{staff.pending_requests}}"></div>
                        <button class="btn btn-warning w-100">Update Leave</button>
                    </form>
                </div>
            </div>
        </div>

        <!-- Performance Card -->
        <div class="col">
            <div class="card shadow-sm">
                <div class="card-header bg-danger text-white">Performance</div>
                <div class="card-body">
                    <form method="post">
                        <input type="hidden" name="module" value="performance">
                        <div class="mb-2">KPI Score: <input type="number" name="kpi_score" class="form-control" value="{{staff.kpi_score}}"></div>
                        <div class="mb-2">Peer Review: <input type="text" name="peer_review" class="form-control" value="{{staff.peer_review}}"></div>
                        <div class="mb-2">Manager Feedback: <input type="text" name="manager_feedback" class="form-control" value="{{staff.manager_feedback}}"></div>
                        <button class="btn btn-danger w-100">Update Performance</button>
                    </form>
                </div>
            </div>
        </div>

    </div>
</div>
</body>
</html>
"""
    return render_template_string(template, staff=staff)
      

# =========================

# =========================
# Attrition Prediction
@app.route('/predict_attrition/<staff_id>')
def predict_attrition(staff_id):
    staff = Staff.query.get(staff_id)
    if not staff: return jsonify({"error":"Staff not found"})
    try:
        model_data = load_attrition_model()
        if not model_data: return jsonify({"error":"Attrition model not ready"})
        model, le = model_data
        month_encoded = le.transform([staff.month])[0] if staff.month else 0
        features = [staff.days_present, staff.days_absent, staff.leave_balance, staff.kpi_score, month_encoded]
        pred = model.predict([features])[0]
        prob = float(model.predict_proba([features])[0][1])
        return jsonify({"prediction":"High Attrition Risk" if pred==1 else "Low Attrition Risk","probability":round(prob,2)})
    except Exception as e:
        return jsonify({"error":str(e)})

# =========================
# Delete Staff
@app.route('/delete/<staff_id>')
def delete_staff(staff_id):
    staff = Staff.query.get(staff_id)
    if staff:
        db.session.delete(staff)
        db.session.commit()
    return redirect('/')

# =========================
# Run App
def run_app():
    app.run("192.168.100.87", port=1343, debug=True, use_reloader=False)

# =========================
# Initialize DB, Add Sample Staff & Train Models
with app.app_context():
    db.create_all()
    if not Staff.query.first():
        s1 = Staff(id="S001", name="Win Lae", base_salary=50000, net_pay=45000, days_present=20, days_absent=5, month="Jan", kpi_score=80)
        s2 = Staff(id="S002", name="Loon", base_salary=60000, net_pay=55000, days_present=18, days_absent=7, month="Jan", kpi_score=90)
        s3 = Staff(id="S003", name="Mike Low", base_salary=40000, net_pay=35000, days_present=3, days_absent=10, month="Jan", kpi_score=40)
        s3 = Staff(id="S005", name="Mike", base_salary=40000, net_pay=35000, days_present=3, days_absent=10, month="Jan", kpi_score=50)
        db.session.add_all([s1,s2,s3])
        db.session.commit()
    load_attendance_model()
    load_attrition_model()

threading.Thread(target=run_app).start()


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://192.168.100.87:1343
Press CTRL+C to quit
192.168.100.87 - - [10/Feb/2026 16:21:04] "GET / HTTP/1.1" 200 -
192.168.100.87 - - [10/Feb/2026 16:21:04] "GET /favicon.ico HTTP/1.1" 404 -
C:\Users\HP\AppData\Local\Temp\ipykernel_8760\4222829989.py:216: LegacyAPIWarning: The Query.get() method is considered legacy as of the 1.x series of SQLAlchemy and becomes a legacy construct in 2.0. The method is now available as Session.get() (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  staff = Staff.query.get(staff_id)
192.168.100.87 - - [10/Feb/2026 16:21:08] "GET /staff/S001 HTTP/1.1" 200 -
192.168.100.87 - - [10/Feb/2026 16:21:13] "GET / HTTP/1.1" 200 -
192.168.100.87 - - [10/Feb/2026 16:21:19] "GET /add_staff HTTP/1.1" 200 -
C:\Users\HP\AppData\Local\Temp\ipykernel_8760\4222829989.py:179: LegacyAPIWarning: The Query.get() method is considered legacy as of the 1.x series of SQLAlchemy and becomes a legacy construct in 2.0. The method is 